# Analysis of 3-Arms Pipeline Outputs

This notebook analyzes the output CSVs from the two pipeline runs that hit Groq API rate limits.

**Goal**: Determine how much usable prediction data we have, and what we can salvage.

## Files to Analyze
1. `data/output/predictions_3arms_with_web.csv.tmp` - Web search pipeline output
2. Any outputs from the technical-only pipeline
3. `data/tavily_cache.jsonl` - Successfully cached web search results (1844 markets)

In [ ]:
import pandas as pd
import json
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

## 1. Load Web Search Pipeline Output

In [ ]:
# Load the web search pipeline output
df_web = pd.read_csv('data/output/predictions_3arms_with_web.csv.tmp')

print(f"Total predictions attempted: {len(df_web):,}")
print(f"Expected: {1844 * 3:,} (1844 markets x 3 arms)\n")

# Check columns
print("Columns:")
print(df_web.columns.tolist())

df_web.head()

## 2. Analyze Success Rate

In [ ]:
# Count successful predictions (no error, has p_yes value)
df_web['is_successful'] = df_web['error'].isna() & df_web['p_yes'].notna()

total = len(df_web)
successful = df_web['is_successful'].sum()
failed = total - successful

print("=" * 60)
print("OVERALL SUCCESS RATE")
print("=" * 60)
print(f"Total predictions:    {total:,}")
print(f"Successful:           {successful:,} ({successful/total*100:.2f}%)")
print(f"Failed (rate-limited): {failed:,} ({failed/total*100:.2f}%)")
print()

In [ ]:
# Break down by arm
print("=" * 60)
print("SUCCESS RATE BY ARM")
print("=" * 60)

for arm in ['baseline', 'volume', 'full_technical']:
    df_arm = df_web[df_web['arm'] == arm]
    arm_success = df_arm['is_successful'].sum()
    arm_total = len(df_arm)
    print(f"{arm:15s}: {arm_success:4,} / {arm_total:,} ({arm_success/arm_total*100:.2f}%)")

print()

## 3. Analyze Error Types

In [ ]:
# Look at error messages
df_errors = df_web[df_web['error'].notna()]

print("=" * 60)
print("ERROR BREAKDOWN")
print("=" * 60)

# Extract error type from error message
def extract_error_type(error_msg):
    if pd.isna(error_msg):
        return None
    if 'rate_limit_exceeded' in error_msg:
        if 'requests per minute (RPM)' in error_msg:
            return 'RPM limit (30 req/min)'
        elif 'tokens per minute (TPM)' in error_msg:
            return 'TPM limit (6000 tok/min)'
        else:
            return 'Rate limit (unspecified)'
    elif 'Parse:' in error_msg:
        return 'Parsing error'
    else:
        return 'Other API error'

df_web['error_type'] = df_web['error'].apply(extract_error_type)

error_counts = df_web['error_type'].value_counts()
print(error_counts)
print()

# Show a few example error messages
print("Example error messages:")
print(df_errors['error'].iloc[0][:300])

## 4. Extract Usable Predictions

In [ ]:
# Get only successful predictions
df_usable = df_web[df_web['is_successful']].copy()

print(f"Usable predictions: {len(df_usable):,}\n")

# Check how many unique markets we have at least one prediction for
unique_markets_with_data = df_usable['market_ticker'].nunique()
print(f"Markets with at least one successful prediction: {unique_markets_with_data}")
print(f"Total markets in dataset: 1,844")
print(f"Coverage: {unique_markets_with_data/1844*100:.1f}%\n")

# Check which arms have data
print("Markets by arm:")
for arm in ['baseline', 'volume', 'full_technical']:
    n_markets = df_usable[df_usable['arm'] == arm]['market_ticker'].nunique()
    print(f"  {arm:15s}: {n_markets} markets")

print()
df_usable.head(10)

## 5. Analyze Prediction Distribution

In [ ]:
if len(df_usable) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    for idx, arm in enumerate(['baseline', 'volume', 'full_technical']):
        df_arm = df_usable[df_usable['arm'] == arm]
        axes[idx].hist(df_arm['p_yes'], bins=50, edgecolor='black', alpha=0.7)
        axes[idx].set_title(f'{arm.title()} (n={len(df_arm)})')
        axes[idx].set_xlabel('Predicted Probability')
        axes[idx].set_ylabel('Count')
        axes[idx].set_xlim(0, 1)
    
    plt.tight_layout()
    plt.show()
    
    # Summary stats
    print("\nPrediction statistics by arm:")
    print(df_usable.groupby('arm')['p_yes'].describe())
else:
    print("No usable predictions to visualize!")

## 6. Check Tavily Cache

In [ ]:
# Load Tavily cache to see what web search data we have
tavily_records = []
with open('data/tavily_cache.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        tavily_records.append(json.loads(line))

print(f"Total Tavily searches cached: {len(tavily_records):,}")

# Check how many succeeded
successful_searches = sum(1 for r in tavily_records if not r.get('tavily_error'))
print(f"Successful searches: {successful_searches:,} ({successful_searches/len(tavily_records)*100:.1f}%)")

# Check average number of results per search
results_counts = [len(r.get('results', [])) for r in tavily_records if not r.get('tavily_error')]
if results_counts:
    print(f"Average results per search: {sum(results_counts)/len(results_counts):.1f}")

print("\nExample Tavily record:")
example = tavily_records[0]
print(f"Market: {example['market_ticker']}")
print(f"Query: {example['query']}")
print(f"Results: {len(example.get('results', []))}")
if example.get('results'):
    print(f"\nFirst result:")
    print(f"  Title: {example['results'][0]['title']}")
    print(f"  URL: {example['results'][0]['url']}")
    print(f"  Content: {example['results'][0]['content'][:200]}...")

## 7. Summary and Recommendations

### Key Findings:
- The Tavily web search stage completed successfully (1844/1844 markets)
- The LLM inference stage hit severe Groq rate limits
- Very few successful predictions due to rate limiting

### Rate Limit Issues:
- **RPM limit**: 30 requests per minute
- **TPM limit**: 6,000 tokens per minute
- With 5,532 total API calls needed, this would take ~184 minutes at 30 RPM (3+ hours)
- But we hit the limit almost immediately

### Solutions to Consider:

#### Option 1: Strict Rate Limiting
- Reduce concurrency from 8 to 1-2
- Add sleep between batches to stay under 30 RPM
- Estimated time: 3-4 hours for full run

#### Option 2: Upgrade Groq Tier
- Check if Groq offers paid tiers with higher limits
- Would dramatically speed up processing

#### Option 3: Use Different LLM Provider
- OpenAI API (gpt-4o-mini or gpt-3.5-turbo)
- Anthropic API (Claude Haiku for speed)
- Higher limits, but costs money

#### Option 4: Batch Processing with Delays
- Process in chunks of 25 predictions (just under 30 RPM)
- Wait 60 seconds between chunks
- More reliable but slower

#### Option 5: Split Across Multiple API Keys
- If you have multiple Groq accounts
- Round-robin between keys
- Doubles/triples throughput

### Recommendation:
**Start with Option 4** (batch processing with delays) as it's the most reliable:
- Set concurrency = 1
- Process 25 markets at a time (25 markets x 3 arms = 75 calls)
- Wait 180 seconds between batches (3 minutes to be safe)
- This gives ~500 calls per hour, ~11-12 hours total
- Add checkpoint/resume capability to save progress

## 8. Save Usable Data

In [ ]:
# Save the usable predictions to a clean CSV
if len(df_usable) > 0:
    output_path = 'data/output/predictions_3arms_web_usable.csv'
    df_usable.to_csv(output_path, index=False)
    print(f"Saved {len(df_usable)} usable predictions to: {output_path}")
else:
    print("No usable predictions to save.")

## 9. Check for Technical Pipeline Output

The technical pipeline (without web search) may have more data since it started earlier.

In [ ]:
# Check if there's a technical-only output file
technical_files = list(Path('data/output').glob('predictions_3arms_technical*'))

if technical_files:
    print(f"Found {len(technical_files)} technical pipeline output file(s):\n")
    for f in technical_files:
        print(f"  {f.name} ({f.stat().st_size / 1024 / 1024:.1f} MB)")
    
    # Load and analyze the first one
    df_tech = pd.read_csv(technical_files[0])
    df_tech['is_successful'] = df_tech['error'].isna() & df_tech['p_yes'].notna()
    
    print(f"\nTechnical pipeline stats:")
    print(f"  Total predictions: {len(df_tech):,}")
    print(f"  Successful: {df_tech['is_successful'].sum():,} ({df_tech['is_successful'].mean()*100:.1f}%)")
    
    # Compare with web pipeline
    print(f"\nComparison:")
    print(f"  Web pipeline success rate: {successful/total*100:.1f}%")
    print(f"  Technical pipeline success rate: {df_tech['is_successful'].mean()*100:.1f}%")
else:
    print("No technical pipeline output files found.")
    print("The technical pipeline may have been interrupted before saving.")